In [ ]:
# CELL 1: Mount Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# CELL 2: Clone hoặc pull code

import os

REPO_DIR = '/content/ai-race-llm'

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin haiyen
    print("✅ Đã pull code mới nhất")
else:
    !git clone -b haiyen https://github.com/nmpogg/ai-race-llm.git {REPO_DIR}
    %cd {REPO_DIR}
    print("✅ Đã clone xong")

!git log --oneline -3  # xem 3 commit gần nhất

In [ ]:
# CELL 3: Link data từ Drive

import os

os.makedirs('data', exist_ok=True)

DATA_DRIVE = '/content/drive/MyDrive/ai-race-data'

folders = [
    'Document_config_data',
    'API_config_data', 
    'test_data',
    'example_data'
]

for folder in folders:
    src = f'{DATA_DRIVE}/{folder}'
    dst = f'./data/{folder}'
    if not os.path.exists(dst):
        os.symlink(src, dst)
        print(f"✅ Linked: {folder}")
    else:
        print(f"⏭️ Đã tồn tại: {folder}")

In [ ]:
# CELL 4: Cài thư viện
!pip install -q torch transformers pandas numpy \
    faiss-cpu sentence-transformers rank-bm25 \
    pymupdf pymupdf4llm openpyxl \
    bitsandbytes>=0.43.0 accelerate>=0.27.0
print("✅ Cài xong thư viện")

In [ ]:
# CELL 4b: Restore corpus.md từ Drive (nếu đã có cache)
import os, shutil, sys
sys.path.insert(0, '/content/ai-race-llm')
os.chdir('/content/ai-race-llm')

CACHE_DIR = f'{DATA_DRIVE}/cached'
corpus_local = './data/markdown/corpus.md'
corpus_cache = f'{CACHE_DIR}/corpus.md'

if os.path.exists(corpus_cache):
    os.makedirs('./data/markdown', exist_ok=True)
    shutil.copy(corpus_cache, corpus_local)
    print(f"✅ Restored corpus.md ({os.path.getsize(corpus_local)/1024/1024:.1f} MB)")
else:
    print("⚠️ Chưa có cache → cần chạy CELL 4d")

In [ ]:
# CELL 4d: Chạy pdf2md - chỉ chạy nếu 4b báo chưa có cache
import sys, os
sys.path.insert(0, '/content/ai-race-llm')
os.chdir('/content/ai-race-llm')

from src.preprocess.pdf2md import process_pdf_data

os.makedirs('./data/markdown', exist_ok=True)

process_pdf_data(
    './data/Document_config_data',
    './data/markdown',
    './data/markdown/corpus.md'
)
print("✅ pdf2md xong! Hãy chạy CELL 4c để lưu cache!")

In [ ]:
# CELL 4c: Lưu corpus.md lên Drive (chỉ chạy 1 lần sau khi 4d xong)
import os, shutil

CACHE_DIR = f'{DATA_DRIVE}/cached'
os.makedirs(CACHE_DIR, exist_ok=True)
shutil.copy('./data/markdown/corpus.md', f'{CACHE_DIR}/corpus.md')
print("✅ Đã lưu corpus.md lên Drive!")

In [ ]:
# # CELL 5: Chạy pipeline

# import sys
# sys.path.insert(0, '/content/ai-race-llm')

# from main import load_service, infer
# load_service()
# infer()

In [ ]:
# # CELL 6: Lưu kết quả về Drive

# import shutil, os

# os.makedirs(f'{DATA_DRIVE}/output', exist_ok=True)
# shutil.copy(
#     '/content/ai-race-llm/submission.csv',
#     f'{DATA_DRIVE}/output/submission.csv'
# )
# print("✅ Đã lưu submission.csv về Drive!")

In [ ]:
# CELL 5b: Build BM25
import sys, os, pickle, re, gc
sys.path.insert(0, '/content/ai-race-llm')
os.chdir('/content/ai-race-llm')

from rank_bm25 import BM25Okapi
os.makedirs('./data/knowledge', exist_ok=True)

def tokenize(t):
    return re.findall(r'\b\w+\b', str(t).lower())

print("Đọc corpus...")
with open('./data/markdown/corpus.md', 'r', encoding='utf-8') as f:
    text = f.read()
print(f"Corpus: {len(text)/1024/1024:.1f} MB")

print("Chunking...")
lines = text.split('\n')
chunks = []
current = []
current_len = 0

for line in lines:
    line = line.strip()
    if not line:
        continue
    current.append(line)
    current_len += len(line)
    if current_len >= 500:
        chunks.append(' '.join(current))
        current = []
        current_len = 0

if current:
    chunks.append(' '.join(current))

print(f"Tổng chunks: {len(chunks)}")
del text, lines
gc.collect()

with open('./data/knowledge/chunks.pkl', 'wb') as f:
    pickle.dump(chunks, f)

print("Building BM25...")
bm25 = BM25Okapi([tokenize(c) for c in chunks])
del chunks
gc.collect()

with open('./data/knowledge/bm25.pkl', 'wb') as f:
    pickle.dump(bm25, f)
del bm25
gc.collect()
print("✅ BM25 xong!")

In [ ]:
# CELL 5c: Build FAISS
import sys, os, pickle, gc, torch, faiss
import numpy as np
from sentence_transformers import SentenceTransformer
sys.path.insert(0, '/content/ai-race-llm')
os.chdir('/content/ai-race-llm')

with open('./data/knowledge/chunks.pkl', 'rb') as f:
    chunks = pickle.load(f)

print(f"Embed {len(chunks)} chunks...")
embed_model = SentenceTransformer("keepitreal/vietnamese-sbert")

embeddings = embed_model.encode(
    chunks,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True
)

dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss.normalize_L2(embeddings)
faiss_index.add(embeddings)
faiss.write_index(faiss_index, './data/knowledge/faiss.index')
print(f"✅ FAISS xong! {faiss_index.ntotal} vectors")

del embed_model, embeddings, chunks
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# CELL 5c_save: Lưu index lên Drive để dùng lại
import shutil, os

CACHE_DIR = f'{DATA_DRIVE}/cached'
os.makedirs(CACHE_DIR, exist_ok=True)

shutil.copytree('./data/knowledge', f'{CACHE_DIR}/knowledge', dirs_exist_ok=True)
print("✅ Đã lưu knowledge/ lên Drive!")

In [ ]:
# CELL 5c_restore: Restore index từ Drive (dùng thay 5b+5c nếu đã có cache)
import shutil, os
sys.path.insert(0, '/content/ai-race-llm')
os.chdir('/content/ai-race-llm')

CACHE_DIR = f'{DATA_DRIVE}/cached'

if os.path.exists(f'{CACHE_DIR}/knowledge'):
    os.makedirs('./data/knowledge', exist_ok=True)
    shutil.copytree(f'{CACHE_DIR}/knowledge', './data/knowledge', dirs_exist_ok=True)
    print("✅ Restored knowledge/ từ Drive!")
else:
    print("⚠️ Chưa có cache index → cần chạy 5b và 5c")

In [ ]:
# CELL 5d: Load service và infer
import sys, os, glob
import pandas as pd

sys.path.insert(0, '/content/ai-race-llm')
os.chdir('/content/ai-race-llm')

from sentence_transformers import SentenceTransformer
from src.llm import LLMService
from src.router import RouterAgent
from src.fewshot import FewShotLoader
from src.retrieval.apiretriever import APIRetriever
from src.agents.api import APIAgent
from src.agents.document import DocAgent
import main

# ── 1. Chuẩn bị đường dẫn ────────────────────────────────────────────────────
# Convert xlsx -> csv để APIRetriever đọc được
xlsx_files = glob.glob('./data/API_config_data/*.xlsx')
if xlsx_files:
    xlsx_path = xlsx_files[0]
    csv_path  = xlsx_path.replace('.xlsx', '.csv')
    pd.read_excel(xlsx_path).to_csv(csv_path, index=False, encoding='utf-8-sig')
    FILE_API_CONFIG = csv_path
else:
    FILE_API_CONFIG = glob.glob('./data/API_config_data/*.csv')[0]

print(f"API config : {FILE_API_CONFIG}")

# Override config trong main
main.INPUT_FILE      = './data/test_data/Test_data.xlsx'
main.OUTPUT_FILE     = './data/output/result.csv'
main.CHECKPOINT_FILE = './data/output/checkpoint.csv'
main.INDEX_DIR       = './data/knowledge'
main.API_CSV         = FILE_API_CONFIG
main.EXAMPLE_DIR     = './data/example_data'
main.USE_ENSEMBLE    = True  # Đặt False nếu muốn chạy nhanh hơn

os.makedirs('./data/output', exist_ok=True)
print(f"Test data  : {main.INPUT_FILE}")

# ── 2. Load tất cả services ───────────────────────────────────────────────────
router, doc_agent, api_agent = main.load_services()

# ── 3. Chạy infer ─────────────────────────────────────────────────────────────
print("\n🚀 Bắt đầu infer...")
main.main()

In [ ]:
# CELL 5d_recover: Chạy tiếp từ checkpoint nếu bị ngắt giữa chừng
# Chỉ cần chạy cell này thay cho 5d khi Colab bị disconnect
import sys, os, glob, pandas as pd

sys.path.insert(0, '/content/ai-race-llm')
os.chdir('/content/ai-race-llm')

CHECKPOINT = './data/output/checkpoint.csv'
OUTPUT     = './data/output/result.csv'

if os.path.exists(CHECKPOINT):
    df_ck = pd.read_csv(CHECKPOINT)
    print(f"✅ Checkpoint: đã có {len(df_ck)} câu.")
    print(f"   Chạy lại CELL 5d — pipeline sẽ tự bỏ qua {len(df_ck)} câu đó.")
else:
    print("⚠️ Không tìm thấy checkpoint. Hãy chạy CELL 5d từ đầu.")

In [ ]:
# # Chạy cell này để debug trước
# import glob, os

# # Xem tất cả file trong data/
# print("=== Tất cả file trong data/ ===")
# for f in glob.glob('./data/**/*', recursive=True):
#     if os.path.isfile(f):
#         print(f)

# # Xem source infer()
# import inspect
# import main
# print("\n=== Source infer() ===")
# print(inspect.getsource(main.infer))

In [ ]:
# CELL 6: Lưu kết quả về Drive
import shutil, os

os.makedirs(f'{DATA_DRIVE}/output', exist_ok=True)

# Lưu cả result và checkpoint
shutil.copy('./data/output/result.csv',
            f'{DATA_DRIVE}/output/result.csv')

if os.path.exists('./data/output/checkpoint.csv'):
    shutil.copy('./data/output/checkpoint.csv',
                f'{DATA_DRIVE}/output/checkpoint.csv')

print("✅ Đã lưu result.csv về Drive!")

# Chạy lần đầu
Cell 1          → Mount Drive
Cell 2          → Git pull
Cell 3          → Link data
Cell 4          → Cài thư viện  ← đã thêm bitsandbytes + accelerate
Cell 4b         → Restore corpus (báo chưa có cache)
Cell 4d         → Chạy pdf2md (~20-30 phút)
Cell 4c         → Lưu corpus lên Drive
Cell 5b         → Build BM25
Cell 5c         → Build FAISS
Cell 5c_save    → Lưu index lên Drive
Cell 5d         → Infer (tự checkpoint mỗi 30 câu)
Cell 6          → Lưu kết quả về Drive

# Chạy lần sau (đã có cache)
Cell 1 → 2 → 3 → 4 → 4b → 5c_restore → 5d → 6

# Nếu bị disconnect giữa chừng khi đang infer
Cell 1 → 2 → 3 → 4 → 4b → 5c_restore → 5d_recover (xem tiến độ) → 5d (chạy tiếp tự động) → 6